In [ ]:
import numpy as np
import random

# 第19章 层次强化学习

## 目录
1. 层次强化学习
2. 选项框架
3. 编程题

---
## 1. 层次强化学习

In [ ]:
"""层次强化学习"""
class HierarchicalAgent:
    """层次智能体"""
    def __init__(self, n_states, n_options):
        self.meta_policy = np.zeros((n_states, n_options))
        self.skills = [np.zeros((n_states, 2)) for _ in range(n_options)]
    
    def select_option(self, state):
        return np.argmax(self.meta_policy[state])
    
    def get_action(self, state, option):
        return np.argmax(self.skills[option][state])

print("层次强化学习: 元策略+技能库")

---
## 2. 选项框架

In [ ]:
class Option:
    """技能选项"""
    def __init__(self, init_state, policy, termination_prob):
        self.init = init_state
        self.policy = policy
        self.termination = termination_prob
    
    def should_terminate(self, state):
        return random.random() < self.termination(state)

class SkillLibrary:
    """技能库"""
    def __init__(self, n_skills):
        self.skills = []
        self.rewards = []
    
    def add_skill(self, skill, reward):
        self.skills.append(skill)
        self.rewards.append(reward)
    
    def get_best_skill(self):
        return np.argmax(self.rewards)

print("选项框架: 启动状态+策略+终止条件")

---
## 3. 编程题

### 编程题1：实现两层层次强化学习

In [ ]:
class TwoLevelAgent:
    """两层层次智能体"""
    def __init__(self, n_states, n_options, n_actions):
        self.n_options = n_options
        self.n_actions = n_actions
        self.meta_q = np.zeros((n_states, n_options))
        self.skill_q = [np.zeros((n_states, n_actions)) for _ in range(n_options)]
        self.gamma = 0.99
    
    def select_option(self, state, epsilon=0.1):
        if random.random() < epsilon:
            return random.randint(0, self.n_options - 1)
        return np.argmax(self.meta_q[state])
    
    def select_action(self, state, option):
        return np.argmax(self.skill_q[option][state])
    
    def update_meta(self, s, o, r, ns):
        self.meta_q[s, o] += 0.1 * (r + self.gamma * np.max(self.meta_q[ns]) - self.meta_q[s, o])
    
    def update_skill(self, option, s, a, r, ns):
        self.skill_q[option][s, a] += 0.1 * (r + self.gamma * np.max(self.skill_q[option][ns]) - self.skill_q[option][s, a])

class MultiStepEnv:
    """多步环境"""
    def __init__(self, size=5):
        self.size = size
        self.pos = 0
    def reset(self): self.pos = 0; return self.pos
    def step(self, action):
        if action == 0: self.pos = max(0, self.pos - 1)
        else: self.pos = min(self.size - 1, self.pos + 1)
        done = self.pos == self.size - 1
        reward = 1.0 if done else -0.01
        return self.pos, reward, done

def train_two_level():
    """训练两层层次智能体"""
    agent = TwoLevelAgent(n_states=5, n_options=2, n_actions=2)
    env = MultiStepEnv()
    rewards_history = []
    
    for ep in range(100):
        s = env.reset()
        option = agent.select_option(s)
        total = 0
        
        for _ in range(20):
            a = agent.select_action(s, option)
            ns, r, done = env.step(a)
            agent.update_skill(option, s, a, r, ns)
            total += r
            if done: break
            
            new_option = agent.select_option(ns)
            if new_option != option:
                agent.update_meta(s, option, total, ns)
                option = new_option
            s = ns
        rewards_history.append(total)
    
    print("两层层次智能体结果:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  收敛回合: {next((i for i, r in enumerate(rewards_history) if r > 0.9), -1)}")

train_two_level()

---

### 编程题2：实现技能发现算法

In [ ]:
class SkillDiscovery:
    """技能发现"""
    def __init__(self, n_states, n_skills, n_actions):
        self.n_skills = n_skills
        self.policies = [np.random.randn(n_states, n_actions) * 0.01 for _ in range(n_skills)]
        self.intrinsic_rewards = np.zeros(n_skills)
    
    def get_action(self, state, skill):
        probs = np.exp(self.policies[skill][state])
        return np.random.choice(len(probs), p=probs/np.sum(probs))
    
    def update_skill(self, skill, rewards):
        self.intrinsic_rewards[skill] += np.mean(rewards)
    
    def get_best_skill(self):
        return np.argmax(self.intrinsic_rewards)

def test_skill_discovery():
    """测试技能发现"""
    skills = SkillDiscovery(n_states=5, n_skills=3, n_actions=2)
    env = MultiStepEnv()
    
    for ep in range(50):
        total_rewards = {i: [] for i in range(skills.n_skills)}
        
        for skill in range(skills.n_skills):
            s = env.reset()
            for _ in range(20):
                a = skills.get_action(s, skill)
                ns, r, done = env.step(a)
                total_rewards[skill].append(r)
                s = ns
                if done: break
            skills.update_skill(skill, total_rewards[skill])
    
    print("技能发现结果:")
    print(f"  最好技能: {skills.get_best_skill()}")
    print(f"  各技能收益: {skills.intrinsic_rewards}")

test_skill_discovery()

---

### 编程题3：实现HIRO算法的简化版本

In [ ]:
class HIRO:
    """HIRO简化版"""
    def __init__(self, n_states, n_options, horizon):
        self.horizon = horizon
        self.meta_policy = np.zeros((n_states, n_options))
        self.low_level = [np.zeros((n_states, 2)) for _ in range(n_options)]
    
    def update_high_level(self, s, o, return_):
        self.meta_policy[s, o] += 0.01 * return_
    
    def update_low_level(self, option, trajectory, rewards):
        for s, a, r in zip(trajectory, [0]*len(trajectory), rewards):
            self.low_level[option][s, a] += 0.1 * r

def test_hiro():
    """测试HIRO"""
    agent = HIRO(n_states=5, n_options=2, horizon=5)
    env = MultiStepEnv()
    rewards_history = []
    
    for ep in range(50):
        s = env.reset()
        option = np.argmax(agent.meta_policy[s])
        total = 0
        traj, rews = [], []
        
        for step in range(20):
            a = np.argmax(agent.low_level[option][s]) if random.random() > 0.1 else random.randint(0, 1)
            ns, r, done = env.step(a)
            traj.append(s)
            rews.append(r)
            s = ns
            total += r
            
            if step % agent.horizon == 0 or done:
                G = sum(rews[-agent.horizon:])
                agent.update_high_level(traj[0], option, G)
                agent.update_low_level(option, traj[-agent.horizon:], rews[-agent.horizon:])
            if done: break
        rewards_history.append(total)
    
    print("HIRO测试:")
    print(f"  最终回报: {rewards_history[-1]}")

test_hiro()

print("\n第19章 层次强化学习 - 编程题完成!")